In [ ]:
import xgboost as xgb
print(xgb.__version__)

3.2.0


In [ ]:
from google.colab import files
uploaded = files.upload()

Saving sample_submission.csv to sample_submission.csv
Saving test.csv to test.csv
Saving train.csv to train.csv


In [ ]:

# ============================================================
import pandas as pd
import numpy as np
from sklearn.ensemble import ExtraTreesRegressor
from google.colab import files

train = pd.read_csv('train.csv')
test  = pd.read_csv('test.csv')

for df in [train, test]:
    df['hr']  = df['timestamp'].apply(lambda x: int(x.split(':')[0]))
    df['mn']  = df['timestamp'].apply(lambda x: int(x.split(':')[1]))
    df['rd']  = df['RoadType'].map({'Residential':0,'Street':1,'Highway':2}).fillna(-1)
    df['lv']  = df['LargeVehicles'].map({'Not Allowed':0,'Allowed':1})
    df['lm']  = df['Landmarks'].map({'No':0,'Yes':1})
    df['wx']  = df['Weather'].map({'Sunny':0,'Rainy':1,'Foggy':2,'Snowy':3}).fillna(-1)
    df['tmp'] = df['Temperature'].fillna(train['Temperature'].median())

BASE32 = '0123456789bcdefghjkmnpqrstuvwxyz'
def dgh(h):
    li=(-90.,90.); lo=(-180.,180.); il=True
    for c in h:
        cd=BASE32.index(c)
        for mask in [16,8,4,2,1]:
            if il: mid=(lo[0]+lo[1])/2; lo=(mid,lo[1]) if cd&mask else (lo[0],mid)
            else:  mid=(li[0]+li[1])/2; li=(mid,li[1]) if cd&mask else (li[0],mid)
            il=not il
    return (li[0]+li[1])/2,(lo[0]+lo[1])/2

for df in [train, test]:
    co = df['geohash'].apply(dgh)
    df['lat'] = co.apply(lambda x: x[0])
    df['lon'] = co.apply(lambda x: x[1])
    df['geo4'] = df['geohash'].str[:4]
    df['geo5'] = df['geohash'].str[:5]

G = train['demand'].mean()

# ── ALL LOOKUP TABLES ────────────────────────────────────────
geo_hrm  = train.groupby(['geohash','hr','mn'])['demand'].mean()  # finest
geo_hr   = train.groupby(['geohash','hr'])['demand'].mean()
geo_m    = train.groupby('geohash')['demand'].mean()
geo_s    = train.groupby('geohash')['demand'].std().fillna(0)
geo_max  = train.groupby('geohash')['demand'].max()
geo_med  = train.groupby('geohash')['demand'].median()
g4_m     = train.groupby('geo4')['demand'].mean()
g5_m     = train.groupby('geo5')['demand'].mean()
hr_m     = train.groupby('hr')['demand'].mean()
rd_hr    = train.groupby([train['rd'],'hr'])['demand'].mean()
rd_lv    = train.groupby([train['rd'],train['lv']])['demand'].mean()
rd_nl_hr = train.groupby([train['RoadType'],train['NumberofLanes'],'hr'])['demand'].mean()
lm_hr    = train.groupby([train['lm'],'hr'])['demand'].mean()

# ── PURE LOOKUP (finest granularity first) ───────────────────
def smart_lookup(g, h, m, rt, nl, rd):
    v = geo_hrm.get((g,h,m))           # exact location+hour+minute
    if v is None: v = geo_hr.get((g,h))         # location+hour
    if v is None: v = rd_nl_hr.get((rt,nl,h))   # road+lanes+hour
    if v is None: v = rd_hr.get((rd,h))          # road+hour
    if v is None: v = geo_m.get(g)              # location only
    if v is None: v = G                          # global mean
    return v

print("Computing lookup predictions...")
tp_lookup = np.array([
    smart_lookup(g,h,m,rt,nl,r)
    for g,h,m,rt,nl,r in zip(
        test['geohash'],test['hr'],test['mn'],
        test['RoadType'],test['NumberofLanes'],test['rd'])
])
tp_lookup = np.clip(tp_lookup, 0, 1)
print(f"Lookup range: {tp_lookup.min():.4f}–{tp_lookup.max():.4f}")

# ── FEATURE ENGINEERING (lookup as a feature too) ────────────
def fe(df):
    d = df.copy()
    d['tc']   = d['hr'] + d['mn']/60
    d['hsin'] = np.sin(2*np.pi*d['hr']/24)
    d['hcos'] = np.cos(2*np.pi*d['hr']/24)
    d['msin'] = np.sin(2*np.pi*d['mn']/60)
    d['mcos'] = np.cos(2*np.pi*d['mn']/60)
    d['lnr']  = d['NumberofLanes'] * (d['rd']+1)
    d['geo_m']   = d['geohash'].map(geo_m).fillna(G)
    d['geo_s']   = d['geohash'].map(geo_s).fillna(0)
    d['geo_max'] = d['geohash'].map(geo_max).fillna(G)
    d['geo_med'] = d['geohash'].map(geo_med).fillna(G)
    d['g4_m']    = d['geo4'].map(g4_m).fillna(G)
    d['g5_m']    = d['geo5'].map(g5_m).fillna(G)
    d['hr_m']    = d['hr'].map(hr_m).fillna(G)
    d['geo_hrm'] = [geo_hrm.get((g,h,m), geo_hr.get((g,h), geo_m.get(g,G)))
                    for g,h,m in zip(d['geohash'],d['hr'],d['mn'])]
    d['geo_hr_'] = [geo_hr.get((g,h), geo_m.get(g,G)) for g,h in zip(d['geohash'],d['hr'])]
    d['rd_hr']   = [rd_hr.get((r,h),G) for r,h in zip(d['rd'],d['hr'])]
    d['rd_lv']   = [rd_lv.get((r,l),G) for r,l in zip(d['rd'],d['lv'])]
    d['rdnlhr']  = [rd_nl_hr.get((rt,nl,h), rd_hr.get((r,h),G))
                    for rt,nl,r,h in zip(d['RoadType'],d['NumberofLanes'],d['rd'],d['hr'])]
    d['lm_hr']   = [lm_hr.get((l,h),G) for l,h in zip(d['lm'],d['hr'])]
    d['ratio']   = d['geo_hrm'] / (d['geo_m']+1e-9)
    d['hr_ratio']= d['geo_hr_'] / (d['hr_m']+1e-9)
    # Include the lookup as a direct feature
    d['lookup']  = [smart_lookup(g,h,m,rt,nl,r)
                    for g,h,m,rt,nl,r in zip(
                        d['geohash'],d['hr'],d['mn'],
                        d['RoadType'],d['NumberofLanes'],d['rd'])]
    return d

FEATS = [
    'lat','lon','hr','mn','tc','hsin','hcos','msin','mcos',
    'rd','NumberofLanes','lv','lm','tmp','wx','lnr',
    'geo_m','geo_s','geo_max','geo_med','g4_m','g5_m','hr_m',
    'geo_hrm','geo_hr_','rd_hr','rd_lv','rdnlhr','lm_hr',
    'ratio','hr_ratio',
    'lookup',   # ← the pure lookup as a feature
]

print("Building features...")
train_fe = fe(train)
test_fe  = fe(test)

X      = train_fe[FEATS].values
y      = train['demand'].values
X_test = test_fe[FEATS].values
print(f"Shape: {X.shape}")

# ── TRAIN MODEL ──────────────────────────────────────────────
print("Training ExtraTrees (500 trees)...")
model = ExtraTreesRegressor(
    n_estimators     = 500,
    max_features     = 0.75,
    min_samples_leaf = 1,
    n_jobs           = -1,
    random_state     = 42,
)
model.fit(X, y)
tp_model = np.clip(model.predict(X_test), 0, 1)
print(f"Model range: {tp_model.min():.4f}–{tp_model.max():.4f}")

# ── BLEND ────────────────────────────────────────────────────
# Model = 88.55, Lookup = 88.43
# Blend slightly toward model
tp_blend = 0.75 * tp_model + 0.25 * tp_lookup
tp_blend = np.clip(tp_blend, 0, 1)

# Save all three — submit the blend first
pd.DataFrame({'Index':test['Index'],'demand':tp_model}).to_csv('sub_model.csv',  index=False)
pd.DataFrame({'Index':test['Index'],'demand':tp_lookup}).to_csv('sub_lookup.csv', index=False)
pd.DataFrame({'Index':test['Index'],'demand':tp_blend}).to_csv('sub_blend.csv',   index=False)

print("\n✓ Saved 3 files. Submit sub_blend.csv first.")
print("If blend < 88.55, submit sub_model.csv")
print(f"Blend range: {tp_blend.min():.4f}–{tp_blend.max():.4f}")

files.download('sub_blend.csv')
files.download('sub_model.csv')

Computing lookup predictions...
Lookup range: 0.0000–1.0000
Building features...
Shape: (77299, 32)
Training ExtraTrees (500 trees)...
Model range: 0.0000–1.0000

✓ Saved 3 files. Submit sub_blend.csv first.
If blend < 88.55, submit sub_model.csv
Blend range: 0.0000–1.0000


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>